# Iris Classification

## Goal
Build a classification model to predict iris species using sepal and petal measurements.

## Requirements Covered
- EDA and visualization of class separability
- Train and compare k-NN, Logistic Regression, and Decision Tree
- Report accuracy, confusion matrix, precision, and recall
- Save the best model using pickle/joblib
- Include example inference code


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import pickle

from pathlib import Path
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support,
    classification_report
)

PLOTS_DIR = Path("../plots")
MODELS_DIR = Path("../models")
PLOTS_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)


## 1. Load Dataset

In [ ]:
df = pd.read_csv("../data/iris.csv")
df.head()

In [ ]:
print("Dataset shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isna().sum())
print("\nClass distribution:")
print(df["species"].value_counts())

## 2. Exploratory Data Analysis

In [ ]:
df.describe()

In [ ]:
counts = df["species"].value_counts()

plt.figure(figsize=(7, 5))
plt.bar(counts.index, counts.values)
plt.title("Class Distribution")
plt.xlabel("Species")
plt.ylabel("Count")
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "class_distribution.png", dpi=180)
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
for species, group in df.groupby("species"):
    plt.scatter(group["petal_length"], group["petal_width"], label=species)

plt.title("Class Separability: Petal Length vs Petal Width")
plt.xlabel("Petal Length")
plt.ylabel("Petal Width")
plt.legend()
plt.tight_layout()
plt.savefig(PLOTS_DIR / "petal_scatter.png", dpi=180)
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
for species, group in df.groupby("species"):
    plt.scatter(group["sepal_length"], group["sepal_width"], label=species)

plt.title("Class Separability: Sepal Length vs Sepal Width")
plt.xlabel("Sepal Length")
plt.ylabel("Sepal Width")
plt.legend()
plt.tight_layout()
plt.savefig(PLOTS_DIR / "sepal_scatter.png", dpi=180)
plt.show()

### EDA Observation
Petal length and petal width show strong separation between species. Iris-setosa is clearly separable, while Iris-versicolor and Iris-virginica have slight overlap.


## 3. Prepare Train-Test Split

In [ ]:
features = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
target = "species"

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

## 4. Train and Compare Models

In [ ]:
models = {
    "k-NN": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier(n_neighbors=5))
    ]),
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, random_state=42))
    ]),
    "Decision Tree": DecisionTreeClassifier(random_state=42, max_depth=4)
}

results = {}
summary_rows = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test, y_pred, average="weighted", zero_division=0
    )
    cv_accuracy = cross_val_score(model, X, y, cv=5, scoring="accuracy").mean()

    results[name] = {
        "model": model,
        "y_pred": y_pred,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "cv_accuracy": cv_accuracy
    }

    summary_rows.append({
        "Model": name,
        "Test Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "5-Fold CV Accuracy": cv_accuracy
    })

metrics_df = pd.DataFrame(summary_rows).sort_values(
    ["Test Accuracy", "F1 Score", "5-Fold CV Accuracy"],
    ascending=False
)

metrics_df.to_csv("../metrics_comparison.csv", index=False)
metrics_df

## 5. Classification Reports

In [ ]:
for name, result in results.items():
    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)
    print(classification_report(y_test, result["y_pred"]))

## 6. Confusion Matrix for Best Model

In [ ]:
best_name = metrics_df.iloc[0]["Model"]
best_model = results[best_name]["model"]
best_pred = results[best_name]["y_pred"]

print("Best Model:", best_name)

labels = sorted(y.unique())
cm = confusion_matrix(y_test, best_pred, labels=labels)

plt.figure(figsize=(6, 5))
plt.imshow(cm)
plt.title(f"Confusion Matrix - {best_name}")
plt.colorbar()
plt.xticks(range(len(labels)), labels, rotation=30, ha="right")
plt.yticks(range(len(labels)), labels)

for i in range(len(labels)):
    for j in range(len(labels)):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "confusion_matrix_best_model.png", dpi=180)
plt.show()

## 7. Metrics Comparison Plot

In [ ]:
x = np.arange(len(metrics_df["Model"]))
width = 0.2

plt.figure(figsize=(8, 5))
plt.bar(x - width * 1.5, metrics_df["Test Accuracy"], width, label="Accuracy")
plt.bar(x - width / 2, metrics_df["Precision"], width, label="Precision")
plt.bar(x + width / 2, metrics_df["Recall"], width, label="Recall")
plt.bar(x + width * 1.5, metrics_df["F1 Score"], width, label="F1")

plt.xticks(x, metrics_df["Model"], rotation=15)
plt.ylim(0, 1.1)
plt.title("Model Metrics Comparison")
plt.ylabel("Score")
plt.legend()
plt.tight_layout()
plt.savefig(PLOTS_DIR / "metrics_comparison.png", dpi=180)
plt.show()

## 8. Save Best Model

In [ ]:
# Fit best model again on training data before saving
best_model.fit(X_train, y_train)

joblib.dump(best_model, MODELS_DIR / "best_iris_model.joblib")

with open(MODELS_DIR / "best_iris_model.pkl", "wb") as f:
    pickle.dump(best_model, f)

print(f"Saved best model: {best_name}")
print("Joblib file:", MODELS_DIR / "best_iris_model.joblib")
print("Pickle file:", MODELS_DIR / "best_iris_model.pkl")

## 9. Example Inference

In [ ]:
loaded_model = joblib.load(MODELS_DIR / "best_iris_model.joblib")

sample = pd.DataFrame(
    [[5.1, 3.5, 1.4, 0.2]],
    columns=features
)

prediction = loaded_model.predict(sample)[0]

print("Sample input:")
display(sample)
print("Predicted species:", prediction)

## Conclusion

The Iris dataset is clean and balanced. Petal measurements provide strong separation between classes. After comparing k-NN, Logistic Regression, and Decision Tree, the best model is saved for future inference.
